In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import gstools as gs
import pymc as pm
import statsmodels.api as sm
import arviz as az
import pytensor.tensor as pt
import pickle
import pathlib
from scipy.spatial.distance import cdist

In [2]:
obs_chosen = pd.read_csv("../data/obs_chosen.csv")
obs_chosen.head()

,Unnamed: 0,ZIP_CODE,STATE,lat,lon,prop_high_school_or_higher_18plus,prop_college_degree_or_higher_18plus,election_diff,log_contrib_2020,log_contrib_2022,log_income,log_population
0,1397,7823,NJ,40.8308,-75.0503,0.947690,0.229131,-1,7.625595,7.488294,11.356529,8.868132
1,1403,7832,NJ,40.9388,-75.0550,0.929081,0.358874,-1,7.147559,7.868254,11.575186,8.242756
2,1446,8002,NJ,39.9308,-75.0175,0.906090,0.459808,-1,9.199381,9.402777,11.522391,10.082470
3,1451,8007,NJ,39.8651,-75.0564,0.933609,0.392033,-1,5.973810,5.686975,11.508566,8.609955
4,1454,8012,NJ,39.7901,-75.0367,0.940267,0.317135,-1,8.169903,8.815518,11.466368,10.560515


In [3]:
# sample split into 80% train, 20% test
train = obs_chosen.sample(frac=0.8, random_state=305)
test = obs_chosen.drop(train.index)

In [25]:
RUN_BAYESIAN_SPATIAL = True
BAYESIAN_TRACE_PATH = pathlib.Path("trace_log_contrib_2022.pkl")

In [26]:
def prepare_spatial_log_contrib_2022_data(
    df,
    covariates=('prop_college_degree_or_higher_18plus', 
                     'election_diff', 'log_contrib_2020', 'log_income', 'log_population'),
    sample_n=None,
    random_state=305,
):
    cols = ["log_contrib_2022", "lat", "lon", *covariates]
    sample = df.dropna(subset=cols).copy().reset_index(drop=True)

    if sample_n is not None and sample_n < len(sample):
        sample = sample.sample(n=sample_n, random_state=random_state).reset_index(drop=True)

    gdf = gpd.GeoDataFrame(
        sample,
        geometry=gpd.points_from_xy(sample["lon"], sample["lat"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:5070")
    coords_km = np.column_stack([gdf.geometry.x, gdf.geometry.y]) / 1000

    X_raw = sample[list(covariates)].astype(float)
    covariate_means = X_raw.mean()
    covariate_sds = X_raw.std(ddof=0).replace(0, 1)

    X_scaled = (X_raw - covariate_means) / covariate_sds
    X = np.column_stack([np.ones(len(sample)), X_scaled.values])
    y = sample["log_contrib_2022"].values.astype(float)

    return {
        "sample": sample,
        "coords_km": coords_km,
        "X": X,
        "y": y,
        "covariates": list(covariates),
        "covariate_means": covariate_means,
        "covariate_sds": covariate_sds,
    }
    

In [27]:
def fit_bayesian_spatial_log_contrib_2022_model(
    df,
    covariates=('prop_college_degree_or_higher_18plus', 
                     'election_diff', 'log_contrib_2020', 'log_income', 'log_population'),
    sample_n=None,
    draws=1000,
    tune=1000,
    chains=4,
    cores=1,
    target_accept=0.9,
    beta_sigma=10.0,
    nugget_ratio_bounds=(1.0, 100.0),
    inv_phi_bounds=(1.0, 2000.0),  # kilometers, after EPSG:5070 projection
    n_prior_samples=100,
    jitter=1e-6,
    **pm_kwargs,
):
    data = prepare_spatial_log_contrib_2022_data(
        df,
        covariates=covariates,
        sample_n=sample_n,
    )

    X = data["X"]
    y = data["y"]
    coords_km = data["coords_km"]
    D = cdist(coords_km, coords_km)
    n = len(y)

    with pm.Model() as model:
        log_sigma2 = pm.Normal("log_sigma2", mu=0, sigma=10)
        sigma2 = pm.Deterministic("sigma2", pm.math.exp(log_sigma2))
        nugget_ratio = pm.Uniform(
            "nugget_ratio",
            lower=nugget_ratio_bounds[0],
            upper=nugget_ratio_bounds[1],
        )
        inv_phi = pm.Uniform(
            "inv_phi",
            lower=inv_phi_bounds[0],
            upper=inv_phi_bounds[1],
        )
        phi = pm.Deterministic("phi", 1.0 / inv_phi)
        beta = pm.Normal("beta", mu=0, sigma=beta_sigma, shape=X.shape[1])

        H = pm.math.exp(-(phi ** 2) * D ** 2)
        Sigma = sigma2 * H + nugget_ratio * sigma2 * pt.eye(n) + jitter * pt.eye(n)
        mu = pm.math.dot(X, beta)
        pm.MvNormal("y_obs", mu=mu, cov=Sigma, observed=y)

        prior = pm.sample_prior_predictive(samples=n_prior_samples)
        trace = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            cores=cores,
            target_accept=target_accept,
            idata_kwargs={"log_likelihood": True},
            return_inferencedata=True,
            **pm_kwargs,
        )
        pm.sample_posterior_predictive(trace, extend_inferencedata=True)

    trace.extend(prior)
    return trace, data


In [ ]:
trace_delta_census = None
bayes_train_data = None

if RUN_BAYESIAN_SPATIAL:
    trace_delta_census, bayes_train_data = fit_bayesian_spatial_log_contrib_2022_model(
        train,
        sample_n = 100,
        covariates = ('prop_college_degree_or_higher_18plus', 
                     'election_diff', 'log_contrib_2020', 'log_income', 'log_population'),
        draws=100,
        tune=100,
        chains=4,
        cores=1,
        target_accept=0.9,
        nugget_ratio_bounds=(1.0, 100.0),
        inv_phi_bounds=(1.0, 2000.0),
        beta_sigma=5.0,
        n_prior_samples=100,
    )
    with open(BAYESIAN_TRACE_PATH, "wb") as f:
        pickle.dump({"trace": trace_delta_census, "data": bayes_train_data}, f)
elif BAYESIAN_TRACE_PATH.exists():
    with open(BAYESIAN_TRACE_PATH, "rb") as f:
        saved = pickle.load(f)
    trace_delta_census = saved["trace"]
    bayes_train_data = saved["data"]


Sampling: [beta, inv_phi, log_sigma2, nugget_ratio, y_obs]
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [log_sigma2, nugget_ratio, inv_phi, beta]


Output()

Sampling 4 chains for 100 tune and 100 draw iterations (400 + 400 draws total) took 15 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [y_obs]


Output()